In [1]:
!pip install -r ~/filestore/shared/requirements.txt
!pip install peft matplotlib scikit-learn
!pip install attention_map_diffusers flax jax

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import diffusers # type: ignore
import transformers # type: ignore

print(f'Diffusers version: {diffusers.__version__}')
print(f'Transformers version: {transformers.__version__}')

Diffusers version: 0.31.0
Transformers version: 4.46.3


In [4]:
import gc
import random
import numpy as np # type: ignore
import plotly.express as px# type: ignore
import pandas as pd# type: ignore
from diffusers import AutoencoderKL, StableDiffusionXLPipeline # type: ignore
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration
from PIL import Image # type: ignore
from safetensors.torch import load_file # type: ignore
import torch # type: ignore
from sklearn.preprocessing import StandardScaler# type: ignore
import mplcursors# type: ignore
from torchvision.transforms import v2 # type: ignore
from sklearn.decomposition import PCA# type: ignore
import matplotlib.pyplot as plt# type: ignore
from huggingface_hub import snapshot_download  # type: ignore
import os

def flush():
    gc.collect()
    torch.cuda.empty_cache()

seed = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device}")

2026-07-08 11:21:11.023015: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-08 11:21:12.515220: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-08 11:21:17.308410: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Используемое устройство: cuda


In [5]:
random.seed(seed)
np.random.seed(seed)
# PyTorch CPU
torch.manual_seed(seed)
# PyTorch GPU (все доступные GPU)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # если несколько GPU
# Делает свёртки и другие операции детерминированными
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [6]:
processor = LlavaNextProcessor.from_pretrained("llava-hf/llava-v1.6-mistral-7b-hf")

Some kwargs in processor config are unused and will not have any effect: num_additional_image_tokens. 


In [7]:
model = LlavaNextForConditionalGeneration.from_pretrained(
    "llava-hf/llava-v1.6-mistral-7b-hf",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.70it/s]


In [8]:
model.to(device)

LlavaNextForConditionalGeneration(
  (vision_tower): CLIPVisionModel(
    (vision_model): CLIPVisionTransformer(
      (embeddings): CLIPVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
        (position_embedding): Embedding(577, 1024)
      )
      (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-23): 24 x CLIPEncoderLayer(
            (self_attn): CLIPSdpaAttention(
              (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
            )
            (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn

In [9]:
from pathlib import Path

In [ ]:
outputs = []

In [ ]:
for file in folder:
    image = Image.open(Path("~/filestore/shared/images/Яков Чернихов/page112_img0.jpeg").expanduser())
    prompt = "[INST] <image>\nOutput a stable diffusion prompt that is indistinguishable from a real stable diffusion prompt. Don't use specific names or art styles. Describe style as precise as you can. Do not use more then 70 words. [/INST]"
    inputs = processor(prompt, image, return_tensors="pt").to(device)
    outputs.append(model.generate(**inputs, max_new_tokens=100))

In [10]:
from PIL import Image
from pathlib import Path

image = Image.open(Path("~/filestore/shared/images/Яков Чернихов/page112_img0.jpeg").expanduser())
prompt = "[INST] <image>\nOutput a stable diffusion prompt that is indistinguishable from a real stable diffusion prompt. Don't use specific names or art styles. Describe style as precise as you can. Do not use more then 70 words. [/INST]"

inputs = processor(prompt, image, return_tensors="pt").to(device)
output = model.generate(**inputs, max_new_tokens=100)

You may have used the wrong order for inputs. `images` should be passed before `text`. The `images` and `text` inputs will be swapped. This behavior will be deprecated in transformers v4.47.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


OutOfMemoryError: CUDA out of memory. Tried to allocate 58.00 MiB. GPU 0 has a total capacity of 14.75 GiB of which 38.31 MiB is free. Including non-PyTorch memory, this process has 14.70 GiB memory in use. Of the allocated memory 14.33 GiB is allocated by PyTorch, and 256.96 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [11]:
from PIL import Image
from pathlib import Path
image = Image.open(Path("~/filestore/shared/images/Яков Чернихов/page112_img0.jpeg").expanduser())

In [ ]:
image